# Imports

In [6]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import numpy.typing as npt
import pandas as pd
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer
from pathlib import Path
from huggingface_hub import login

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Data Loading

In [7]:
DATA_DIR = Path.cwd().parent / "data"
SUB_DIR = Path.cwd().parent / "submissions"

INTERACTIONS_PATH = DATA_DIR / "interactions_train.csv"
ITEMS_PATH = DATA_DIR / "items.csv"
ENRICHED_ITEMS_PATH = DATA_DIR / "enriched_items_merge_openlibrary_googlebooksAPI.csv" 
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

interactions = pd.read_csv(INTERACTIONS_PATH)
interactions.columns = ['u', 'i', 't']
items = pd.read_csv(ITEMS_PATH)
df = pd.read_csv(ENRICHED_ITEMS_PATH)

n_users = interactions['u'].nunique()
n_items = items['i'].nunique()

with open(DATA_DIR / "hf_login.txt", "r") as f:
    token = f.read()
    
login(token=token)

# Feature Extraction - Text Processing & Embeddings

In [8]:
MODEL_CHOICE = "e5" 

model_mapping = {
    "minilm": "paraphrase-multilingual-MiniLM-L12-v2",
    "e5": "intfloat/multilingual-e5-base",
    "gemma": "google/embeddinggemma-300m"
}

print(f"Preparing Text Features using {model_mapping[MODEL_CHOICE]}")
rich_texts = []
for _, row in df.iterrows():
    title = str(row.get('Title', row.get('title', ''))).strip()
    author = str(row.get('Author', row.get('author', ''))).strip()
    subjects = str(row.get('Subjects', row.get('subjects', ''))).strip() 
    genres = str(row.get('genres', '')).strip()
    summary = str(row.get('summary', '')).strip()
    
    parts = []
    if title: parts.append(f"Title: {title}.")
    if author: parts.append(f"Author: {author}.")
    if subjects: parts.append(f"Subjects: {subjects}.")
    if genres: parts.append(f"Genres: {genres}.")
    if summary: parts.append(f"Summary: {summary}.")
    
    full_text = " ".join(parts) if parts else "Unknown book."
    
    # E5 models require a prefix for document indexing
    if MODEL_CHOICE == "e5":
        full_text = "passage: " + full_text
        
    rich_texts.append(full_text)

# Encode Texts
embedder = SentenceTransformer(model_mapping[MODEL_CHOICE], device=device)
embeddings = embedder.encode(rich_texts, show_progress_bar=True, convert_to_numpy=True)

Preparing Text Features using intfloat/multilingual-e5-base


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/478 [00:00<?, ?it/s]

# Matrix-blending

In [9]:
def create_data_mtx(data : pd.DataFrame, n_users: int) -> npt.NDArray[np.float64]:
    """
    Create a user-item interaction matrix from the given data.

    :param data: A DataFrame containing user-item interactions with columns "u" for user IDs and "i" for item IDs.
    :type data: pd.DataFrame
    :param n_users: The number of users.
    :type n_users: int
    :param n_items: The number of items.
    :type n_items: int
    :return: A user-item interaction matrix.
    :rtype: npt.NDArray[np.float64]
    """
    # data_mtx = np.zeros((n_users, n_items))    # Ensure the matrix can accommodate all item IDs
    # data_mtx[data["u"].values, data["i in order"].values] = 1
    data_mtx = np.zeros((n_users, max(interactions.i.unique())+1))
    data_mtx[data["u"].values, data["i"].values] = 1
    return data_mtx

def item_based_predict(interactions : pd.DataFrame, similarity : npt.NDArray[np.float64], epsilon=1e-9):
    """
    Predicts user-item interactions based on item-item similarity.

    :param interactions: The user-item interaction matrix.
    :type interactions: pd.DataFrame
    :param similarity: The item-item similarity matrix.
    :type similarity: npt.NDArray[np.float64]
    :param epsilon: Small constant added to the denominator to avoid division by zero.
    :type epsilon: float
    :return: The predicted interaction scores for each user-item pair.
    :rtype: npt.NDArray[np.float64]
    """
    # np.dot does the matrix multiplication. Here we are calculating the
    # weighted sum of interactions based on item similarity
    pred = similarity.dot(interactions.T) / (similarity.sum(axis=1)[:, np.newaxis] + epsilon)
    return pred.T  # Transpose to get users as rows and items as columns

# Create the full Interaction Matrix (just like the original notebook)
full_data_mtx = create_data_mtx(interactions, n_users)

# Calculate CF Item-Item Similarity (The baseline)
print("Calculating Collaborative Filtering similarities")
cf_item_similarity = cosine_similarity(full_data_mtx.T)

# Calculate Text Similarity (Your Hugging Face NLP power)
print("Calculating Text Semantic similarities")
# embeddings is the numpy array generated by your SentenceTransformer
text_item_similarity = cosine_similarity(embeddings)

# Blend them together! (Alpha is the weight of CF vs Text)
alpha = 0.3
print(f"Blending matrices with CF weight = {alpha}")
hybrid_item_similarity = (alpha * cf_item_similarity) + ((1 - alpha) * text_item_similarity)

# Make Predictions using the Hybrid Matrix
print("Generating Hybrid Predictions")
# item_based_predict is the function from your original lab notebook
hybrid_prediction = item_based_predict(full_data_mtx, hybrid_item_similarity)

# Generate Submission
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)
predictions_item = []

for u in sample_submission["user_id"]:
    # Copy the scores for the user
    u_scores = hybrid_prediction[u, :].copy()
    
    # Get the top 10 highest scoring unread items
    top_10_items = np.argsort(u_scores)[-10:][::-1]
    
    predictions_item.append(" ".join(map(str, top_10_items)))

sample_submission["recommendation"] = predictions_item

out_path = SUB_DIR / f"submission_hybrid_matrix_{MODEL_CHOICE}_alpha_0{int(alpha*100)}.csv"
sample_submission.to_csv(out_path, index=False)
print(f"Success! Saved to {out_path}")

Calculating Collaborative Filtering similarities
Calculating Text Semantic similarities
Blending matrices with CF weight = 0.3
Generating Hybrid Predictions
Success! Saved to c:\Users\Julien\OneDrive\VS Code\OneDrive\GitHub\library-recommender\submissions\submission_hybrid_matrix_e5_alpha_030.csv


In [10]:
# 1. Save the hybrid similarity matrix
np.save(f"hybrid_item_similarity_{MODEL_CHOICE}.npy", hybrid_item_similarity)

# 2. Save a lightweight version of the item catalog for the UI
ui_catalog = df[['i', 'Title', 'Author']].drop_duplicates(subset=['i'])
ui_catalog.to_parquet("item_catalog.parquet", index=False)